In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df = pd.read_csv("IMDB Dataset.csv")

df['sentiment'] = df['sentiment'].astype(str)
df['sentiment'] = df['sentiment'].str.strip().str.lower()

# SAFE mapping
df['sentiment'] = df['sentiment'].apply(
    lambda x: 1 if x == 'positive' else 0
)

print(df['sentiment'].unique())

[1 0]


In [ ]:
#Clean text
import re

def clean_text(text):
    text = text.lower()
    text = text.replace("<br />", " ")
    return text

df['review'] = df['review'].apply(clean_text)

In [ ]:
#Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['review'])

X = tokenizer.texts_to_sequences(df['review'])

In [ ]:
#Padding
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(X, maxlen=200)

import numpy as np
X = np.array(X, dtype=np.int32)

In [ ]:
#Train-test split
from sklearn.model_selection import train_test_split

y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#RNN Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()

model.add(Embedding(5000, 64, input_length=200))
model.add(SimpleRNN(32))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
#Train model
model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.6605 - loss: 0.6046 - val_accuracy: 0.6909 - val_loss: 0.5832
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - accuracy: 0.7922 - loss: 0.4493 - val_accuracy: 0.8363 - val_loss: 0.3905
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.8523 - loss: 0.3500 - val_accuracy: 0.8133 - val_loss: 0.4296
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 28ms/step - accuracy: 0.9050 - loss: 0.2407 - val_accuracy: 0.8260 - val_loss: 0.4200
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 30ms/step - accuracy: 0.9464 - loss: 0.1519 - val_accuracy: 0.8322 - val_loss: 0.4720


In [ ]:
#Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8425 - loss: 0.4414
Accuracy: 0.8424999713897705
